# LightOnOCR-2 Magyar Fine-tuning (v3)

**Futtatás előtt:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# 1. Telepítés + Magyar font letöltése
!pip install -q transformers>=4.45.0 peft datasets accelerate pillow

# Noto Sans font - támogatja a magyar karaktereket
!wget -q https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Regular.ttf -O /content/NotoSans-Regular.ttf
!wget -q https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Bold.ttf -O /content/NotoSans-Bold.ttf

print("✓ Telepítés kész, font letöltve")

In [ ]:
# 2. Tanító adatok generálása magyar fonttal
import json
import random
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

# Font betöltése
FONT_PATH = "/content/NotoSans-Regular.ttf"

# Teszteljük a fontot
test_font = ImageFont.truetype(FONT_PATH, 24)
test_img = Image.new("RGB", (400, 50), "white")
test_draw = ImageDraw.Draw(test_img)
test_draw.text((10, 10), "őűŐŰ öüóőúéáűí ÖÜÓŐÚÉÁŰÍ", fill="black", font=test_font)
display(test_img)
print("↑ Ha látod az ékezeteket, a font működik!")

In [ ]:
# 3. Adatgenerálás
HUNGARIAN_WORDS = [
    # ő karakterek
    "őr", "őriz", "ők", "ősz", "ősi", "őszinte", "őrült",
    "erő", "idő", "mező", "tető", "fő", "nő", "bő", "hő",
    "belső", "külső", "felső", "alsó", "utolsó", "első",
    "költő", "festő", "vezető", "Győr", "tükörfúrógép",
    "Csatornadíj", "vízdíj", "díj", "dőlt", "dől",
    # ű karakterek  
    "űr", "űrlap", "gyűrű", "tűz", "fűz", "gyűjt",
    "hűtő", "hűvös", "hűség", "szürke", "szűk",
    "halványszürke", "fizetendő", "összeg", "adószám",
    "Árvíztűrő", "ÁRVÍZTŰRŐ",
]

def generate_text():
    words = random.sample(HUNGARIAN_WORDS, min(8, len(HUNGARIAN_WORDS)))
    lines = [" ".join(words)]
    lines.append(f"Fizetendő összeg: {random.randint(1,99)} {random.randint(100,999):03d} Ft")
    lines.append(f"Csatornadíj: {random.randint(1,9)} {random.randint(100,999):03d} Ft")
    lines.append(f"Vízdíj alapdíj: {random.randint(1,9)} {random.randint(100,999):03d} Ft")
    lines.append(f"Adószám: {random.randint(10000000,99999999)}-{random.randint(1,2)}-{random.randint(10,99)}")
    lines.append("öüóőúéáűí - ÖÜÓŐÚÉÁŰÍ")
    lines.append("Halványszürke szöveg, dőlt betűk")
    lines.append("Árvíztűrő tükörfúrógép - ÁRVÍZTŰRŐ TÜKÖRFÚRÓGÉP")
    return "\n".join(lines)

def render_text(text, width=750, font_size=24):
    font = ImageFont.truetype(FONT_PATH, font_size)
    lines = text.split("\n")
    line_height = font_size + 12
    height = len(lines) * line_height + 60
    
    # Véletlenszerű háttér
    bg_color = random.choice(["white", "#fafafa", "#f5f5f5", "#fffef0"])
    img = Image.new("RGB", (width, height), bg_color)
    draw = ImageDraw.Draw(img)
    
    y = 30
    for line in lines:
        draw.text((30, y), line, fill="black", font=font)
        y += line_height
    return img

# Generálás
Path("training_data/images").mkdir(parents=True, exist_ok=True)
annotations = []

NUM_SAMPLES = 200
for i in range(NUM_SAMPLES):
    text = generate_text()
    img = render_text(text, font_size=random.choice([20, 22, 24, 26, 28]))
    img.save(f"training_data/images/{i:05d}.png")
    annotations.append({"image": f"{i:05d}.png", "text": text})
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{NUM_SAMPLES}")

with open("training_data/annotations.jsonl", "w", encoding="utf-8") as f:
    for a in annotations:
        f.write(json.dumps(a, ensure_ascii=False) + "\n")

print(f"\n✓ Generálva: {NUM_SAMPLES} kép")

# Példa megjelenítése
print("\nPélda kép:")
display(Image.open("training_data/images/00000.png"))

In [ ]:
# 4. Modell betöltése
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "lightonai/LightOnOCR-2-1B-base"

print(f"Modell betöltése: {MODEL_ID}")
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"✓ Modell betöltve")
print(f"  Device: {model.device}")
print(f"  Paraméterek: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 5. LoRA konfiguráció
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 6. Dataset
import json
from datasets import Dataset
from PIL import Image

def load_data():
    data = []
    with open("training_data/annotations.jsonl", encoding="utf-8") as f:
        for line in f:
            entry = json.loads(line)
            data.append({
                "image_path": f"training_data/images/{entry['image']}",
                "text": entry["text"]
            })
    return Dataset.from_list(data)

def process_example(example):
    image = Image.open(example["image_path"]).convert("RGB")
    text = example["text"]
    
    image_inputs = processor.image_processor(image, return_tensors="pt")
    text_inputs = processor.tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        max_length=512,
        truncation=True,
    )
    
    return {
        "pixel_values": image_inputs["pixel_values"].squeeze(0),
        "input_ids": text_inputs["input_ids"].squeeze(0),
        "attention_mask": text_inputs["attention_mask"].squeeze(0),
        "labels": text_inputs["input_ids"].squeeze(0),
    }

dataset = load_data()
processed_dataset = dataset.map(process_example, remove_columns=dataset.column_names)
print(f"✓ Dataset: {len(processed_dataset)} példa")

In [ ]:
# 7. Training
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./lighton-hun-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    bf16=True,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
)

print("Tanítás indítása...")
trainer.train()
print("\n✓ Tanítás kész!")

In [ ]:
# 8. Mentés
print("LoRA adapter mentése...")
model.save_pretrained("./lighton-hun-lora")

print("LoRA súlyok összefésülése...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./lighton-hun-merged")
processor.save_pretrained("./lighton-hun-merged")

print("✓ Modell mentve: ./lighton-hun-merged")

In [ ]:
# 9. Teszt
print("Teszt futtatása...")
test_img = Image.open("training_data/images/00000.png")
display(test_img)

inputs = processor.image_processor(test_img, return_tensors="pt")
inputs = {k: v.to(merged_model.device) for k, v in inputs.items()}
inputs["input_ids"] = processor.tokenizer("", return_tensors="pt")["input_ids"].to(merged_model.device)

with torch.no_grad():
    outputs = merged_model.generate(**inputs, max_new_tokens=300, do_sample=False)

result = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n" + "="*50)
print("OCR EREDMÉNY:")
print("="*50)
print(result)

In [ ]:
# 10. Letöltés
!zip -r lighton-hun-merged.zip lighton-hun-merged/

from google.colab import files
files.download("lighton-hun-merged.zip")

print("\n" + "="*60)
print("KÖVETKEZŐ LÉPÉS MAC-EN:")
print("="*60)
print("")
print("1. Kicsomagolás:")
print("   unzip lighton-hun-merged.zip")
print("")
print("2. MLX konverzió:")
print("   mlx_vlm convert --hf-path lighton-hun-merged \\")
print("       --mlx-path models/lighton-hun-mlx -q --q-bits 4")
print("")
print("3. Használat:")
print("   python run_ocr.py test.pdf --engine lighton \\")
print("       --model models/lighton-hun-mlx")